# Part 3: Measurement of the Z-boson mass and decay width.

## Fitting the Z mass peak

In [Part 1](./Part1_Analysis_workflow.ipynb) you selected muon pairs compatible with Z-boson decay, and in [Part 2](./Part2_Muon_efficiencies.ipynb) you measured the effect of selection requirements. Now fit the selected invariant-mass spectrum to estimate the Z-boson mass and width.

We have implemented two different functions usually used for fitting the Z mass peak: a Gaussian and a Breit-Weigner. 

* [Gaussian](https://en.wikipedia.org/wiki/Normal_distribution) distribution:

$$ G(x;\mu,\sigma) = \dfrac{1}{\sqrt{2\pi}\sigma}\exp[-\dfrac{(x-\mu)^{2}}{2\sigma^{2}}]    $$

* [Relativistic Breit-Wigner](https://en.wikipedia.org/wiki/Relativistic_Breit%E2%80%93Wigner_distribution) distribution:

$$ B(m;M,\Gamma)= \dfrac{k}{(m^{2}-M^{2})^{2} + M^{2}\Gamma^{2}} ;\hspace{1cm} k = \dfrac{2\sqrt{2}M\Gamma\gamma}{\pi\sqrt{M^{2}+\gamma}} ;\hspace{1cm} \gamma = \sqrt{M^{2}(M^{2}+\Gamma^{2}}   $$

To fit a generator-level Z peak a Breit-Wigner fit makes sense. However, reconstructed-level Z peaks have many detector resolutions that smear the Z mass peak. If the detector resolution is relatively poor, then it is usually good enough to fit a gaussian (since the gaussian detector resolution will overwhelm the inherent Briet-Wigner shape of the peak). If the detector resolution is fairly good, then another option is to fit a Breit-Wigner (for the inherent shape) convoluted with a gaussian (to describe the detector effects).This is in the "no-background" case. If you have backgrounds in your sample (Drell-Yan, cosmics, etc...), and you want to do the fit over a large mass range, then another function needs to be included to take care of this - an exponential is commonly used. 

**Connection with Part 2:** Muon identification and isolation cuts improve the purity of the selected sample, but also reduce the event yield. In this part, fit the selected Z candidate spectrum and consider the efficiency–purity trade-off when interpreting your result.

In [ ]:
cd ~/work/CmsOpenData/RDFAnalyzer

## 1. Import ROOT and necessary tools

In [ ]:
# Import ROOT (ROOT is a analysis package used to handle 
# the tree files, but also histograms, etc.)
import ROOT as R
%jsroot off
from utils import *

In [ ]:
R.gROOT.ProcessLine('.L ./_tdrstyle.C')
R.setTDRStyle()
R.gROOT.SetBatch(1)

## 2. Load histograms

We are going to load the invariant mass histogram that we produced in the previous part.

In [ ]:
fread = R.TFile.Open("../temp/hists_selection.root","READ")
h_sel = fread.Get(f"h_DiMuon_invMass")
h_sel.SetDirectory(0)
fread.Close()

## 3. Fit the invariant mass distribution

To fit the Z peak, we are going to use the custom function _plotFit()_. On the back end, it uses ROOT libraries to perform the fit to the histogram. _plotFit()_ takes the following arguments:
* c          : TCanvas to draw the histogram with the fit
* h          : TH1F histogram that we want to fit
* **fit        : type of fit that we want to do. Options:**
    * **'gaussian' : Gaussian fit**
    * **'bw'       : Relativistic Breit-Wigner fit**
    * **'conv'     : Fit to a convolution between gaussian and Breit-Wigner.**
* color      : plot color
* fit_range  : range in X axis to perform the fit
* logy       : option to plot Y axis in logarithmic scale. Default: true
* xmin, xmax : plot limits of X axis

The code to obtain the gaussian fit is left as an example. You will have to adapt it to fit the other functions.

### 3.1 Gaussian fit

In [ ]:
R.gROOT.FindObject(f"c_gaussian") and R.gROOT.FindObject(f"c_gaussian").Close() # Avoid memory conflicts and crashes
c = R.TCanvas(f"c_gaussian","")
h = h_sel.Clone()
h.SetDirectory(0)
c = plotFit(c, h, 
            fit="gaussian",
            color=R.kOrange+1,
            fit_range=[60.,120.],
            params=[1000.,1.,90.],
            logy=False,
            xmin=61,
            xmax=131)
c.Draw()

### 3.2 Breit-Wigner fit

In [ ]:
# ---------------------------------
# - WRITE YOUR CODE HERE
# ---------------------------------

### 3.3 Convolution fit

In [ ]:
# ---------------------------------
# - WRITE YOUR CODE HERE
# ---------------------------------

**Checkpoint:** Why are simulated events useful even when we already have collision data? What could explain differences between the two Z-peak distributions?


## 4. Real CMS data versus Drell–Yan simulation

Real collision data can be compared with simulated events processed with the **same muon selection**. Here we compare the shapes near the Z peak after normalizing both histograms to the same area. The MC includes generator weights, but this is **not** a luminosity-normalized background prediction.

The data sample originates from a MET primary dataset, so its trigger acceptance may differ from the DY sample. Treat this as an illustrative comparison, not a precision validation.


In [ ]:
import os

# Load the selection from Part 1
with open('../temp/dimuon_selection.txt') as f:
    cutstring = f.read().strip()

# Load MC samples and produce histograms
samples = {'DY': DYMC_PATH, 'TT': TTMC_PATH}
mass = histpars['DiMuon_invMass']
mc_hists = {}

for name, path in samples.items():
    df = loadRDF('Events', os.path.expanduser(path))
    columns = set(str(c) for c in df.GetColumnNames())
    selection = cutstring
    if 'Muon_tkRelIso' in selection and 'Muon_tkRelIso' not in columns:
        if 'Muon_pfRelIso04_all' not in columns:
            raise ValueError(f'{name}: missing muon isolation branch')
        selection = selection.replace('Muon_tkRelIso', 'Muon_pfRelIso04_all')
        print(f'{name}: using PF isolation instead of tracker isolation')
    df = df.Define('Muon_isGoodMuon', f'isGoodMuon({selection})')
    df = df.Define('SelDiMuon_invMass', 'invariantMass(Muon_isGoodMuon, Muon_pt, Muon_eta, Muon_phi)')
    df = df.Filter('SelDiMuon_invMass>=0.')
    h = df.Histo1D((f'h_{name}_MC', '', mass['bins'], mass['xmin'], mass['xmax']), 'SelDiMuon_invMass', 'genWeight').GetValue()
    h.SetDirectory(0)
    integral = h.Integral(h.FindBin(60. + 1e-6), h.FindBin(120. - 1e-6))
    if integral <= 0:
        raise ValueError(f'{name}: empty or non-positive histogram integral in 60-120 GeV')
    h.Scale(1. / integral)
    mc_hists[name] = h

# Copy and normalize data histogram
h_data = h_sel.Clone('h_data_comparison')
h_data.SetDirectory(0)
if h_data.Integral() <= 0:
    raise ValueError('Empty data histogram')
h_data.Scale(1. / h_data.Integral())

# Plot Data vs MC
R.gROOT.FindObject('c_data_mc') and R.gROOT.FindObject('c_data_mc').Close()
c = R.TCanvas('c_data_mc', '', 850, 620)

colors = {'DY': R.kBlue+1, 'TT': R.kRed+1}
h_data.SetStats(0)
h_data.SetTitle(';m_{#mu#mu} (GeV);Normalized events / bin')
h_data.GetXaxis().SetRangeUser(60., 120.)
h_data.SetMaximum(1.35 * max([h_data.GetMaximum()] + [h.GetMaximum() for h in mc_hists.values()]))
h_data.SetMarkerStyle(20)
h_data.SetMarkerSize(0.7)
h_data.SetMarkerColor(R.kBlack)
h_data.SetLineColor(R.kBlack)
h_data.Draw('E')

for name, h in mc_hists.items():
    h.SetStats(0)
    h.SetLineColor(colors[name])
    h.SetLineWidth(2)
    h.Draw('HIST SAME')
h_data.Draw('E SAME')

legend = R.TLegend(0.62, 0.72, 0.89, 0.89)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.AddEntry(h_data, 'CMS data', 'lep')
legend.AddEntry(mc_hists['DY'], 'DY MC', 'l')
legend.AddEntry(mc_hists['TT'], 't#bar{t} MC', 'l')
legend.Draw()

c.Draw()
c.SaveAs('../temp/Z_data_vs_MC.png')

**Checkpoint:** Why are simulated events useful even when we already have collision data? What could explain differences between the two Z-peak distributions?
